In [1]:
from triplet_extraction.src.db.mongo import init_mongo

mongo = init_mongo()
db = mongo["KB_PROPERTY_LAW"]

documents_col = db["documents"]
sections_col = db["legal_sections"]
section_relations_col = db["legal_section_relations"]

You successfully connected to MongoDB!


In [3]:
amended_so_hieu = sections_col.distinct(
    "so_hieu",
    {"is_amendment": True}
)

result = sections_col.delete_many(
    {"so_hieu": {"$in": amended_so_hieu}}
)

print("Deleted documents:", result.deleted_count)

Deleted documents: 22794


In [20]:
from collections import Counter

paths = [
    doc["full_path"]
    for doc in sections_col.find(
        {"full_path": {"$exists": True}, "is_phu_luc": False },
        {"full_path": 1}
    )
]
counter = 0
for path, count in Counter(paths).items():
    if count > 1:
        counter += 1
        print(f"{count} × {path}")
print("Total duplicated paths:", counter)

2 × 09/2025/TT-BXD_điều 2_khoản 5
2 × 09/2025/TT-BXD_điều 2_khoản 5_điểm a
2 × 09/2025/TT-BXD_điều 2_khoản 5_điểm b
2 × 09/2025/TT-BXD_điều 3_khoản 2
2 × 10/2024/TT-BTNMT_chương iii_mục 2_điều 36_khoản 1_điểm a
2 × 10/2024/TT-BTNMT_chương iii_mục 2_điều 36_khoản 1_điểm b
2 × 101/2024/NĐ-CP_chương iii_mục 5
2 × 103/2024/NĐ-CP_chương ii_mục 1_điều 14_khoản 1_điểm a
2 × 103/2024/NĐ-CP_chương ii_mục 1_điều 14_khoản 1_điểm b
2 × 103/2024/NĐ-CP_chương iii_mục 1_điều 36_khoản 1_điểm a
2 × 103/2024/NĐ-CP_chương iii_mục 1_điều 36_khoản 1_điểm b
2 × 103/2024/NĐ-CP_chương iii_mục 1_điều 36_khoản 1_điểm c
2 × 104/2024/NĐ-CP_chương iii_điều 15_khoản 1
2 × 104/2024/NĐ-CP_chương iii_điều 15_khoản 2
2 × 115/2024/NĐ-CP_chương x_điều 66_khoản 2
2 × 115/2024/NĐ-CP_chương x_điều 66_khoản 3
2 × 123/2024/NĐ-CP_chương ii_điều 12_khoản 1_điểm a
2 × 123/2024/NĐ-CP_chương ii_điều 12_khoản 1_điểm b
2 × 123/2024/NĐ-CP_chương ii_điều 12_khoản 1_điểm c
2 × 123/2024/NĐ-CP_chương ii_điều 12_khoản 1_điểm đ
2 × 151/202

In [12]:
doc = sections_col.find(
    {
        "is_amendment": True,
        "type": "điều"
    }
)

694fb816aedc69db48c7134b sửa đổi, bổ sung một số điều của luật quy hoạch số 21/2017/qh14 đã được sửa đổi, bổ sung một số điều theo Luật số 15/2023/QH15, Luật số 16/2023/QH15 và Luật số 28/2023/QH15
694fb816aedc69db48c71355 sửa đổi, bổ sung khoản 4 điều 44 của luật thủy sản số 18/2017/qh14 “4. Thời hạn giao khu vực biển để nuôi trồng thủy sản không quá 50 năm, được tính từ ngày quyết định giao khu vực biển có hiệu lực. Khi hết thời hạn giao, tổ chức, cá nhân có nhu cầu tiếp tục sử dụng khu vực biển đã được giao để nuôi trồng thủy sản được Nhà nước xem xét gia hạn có thể gia hạn nhiều lần nhưng tổng thời gian gia hạn không quá 20 năm. Thời hạn giao khu vực biển cho tổ chức, cá nhân Việt Nam thực hiện nhiệm vụ khoa học và công nghệ phục vụ nuôi trồng thủy sản không quá thời hạn nhiệm vụ khoa học và công nghệ được cơ quan có thẩm quyền phê duyệt.”.
694fb816aedc69db48c71356 sửa đổi, bổ sung một số điều của luật tổ chức chính quyền địa phương số 77/2015/QH13 đã được sửa đổi, bổ sung một số đ

In [14]:
pipeline = [
    {
        "$match": {
            "is_amendment": True,
            "type": "điều"
        }
    },
    {
        "$lookup": {
            "from": "legal_sections",
            "localField": "_id",
            "foreignField": "parent_id",
            "as": "children"
        }
    }
]

results = list(sections_col.aggregate(pipeline))

for dieu in results:
    print(f"\nĐIỀU: {dieu.get('title')}")
    for c in dieu["children"]:
        print("  -", c.get("title"))

    if len(dieu["children"]) == 0:



ĐIỀU: điều 243
  - khoản 1
  - khoản 2
  - khoản 3
  - khoản 4

ĐIỀU: điều 244

ĐIỀU: điều 245
  - khoản 1
  - khoản 2
  - khoản 3
  - khoản 4
  - khoản 5

ĐIỀU: điều 246

ĐIỀU: điều 247

ĐIỀU: điều 248
  - khoản 1
  - khoản 2
  - khoản 3
  - khoản 4
  - khoản 5
  - khoản 6
  - khoản 7
  - khoản 8
  - khoản 9

ĐIỀU: điều 249

ĐIỀU: điều 250

ĐIỀU: điều 251
  - khoản 1
  - khoản 2

ĐIỀU: điều 286

ĐIỀU: điều 305
  - khoản 1
  - khoản 2
  - khoản 3
  - khoản 4

ĐIỀU: điều 377
  - khoản 1
  - khoản 2
  - khoản 3

ĐIỀU: điều 392

ĐIỀU: điều 417

ĐIỀU: điều 421
  - khoản 1
  - khoản 2
  - khoản 3

ĐIỀU: điều 640
  - khoản 1
  - khoản 2
  - khoản 3

ĐIỀU: điều 1
  - khoản 1
  - khoản 2
  - khoản 3
  - khoản 4

ĐIỀU: điều 2

ĐIỀU: điều 3

ĐIỀU: điều 4

ĐIỀU: điều 1
  - khoản 1
  - khoản 2
  - khoản 4
  - khoản 3
  - khoản 4
  - khoản 5
  - khoản 6
  - khoản 7
  - khoản 8
  - khoản 9
  - khoản 10
  - khoản 11
  - khoản 12
  - khoản 3
  - khoản 13
  - khoản 14
  - khoản 15
  - khoản 4
  - khoả